# Multi-Seed RACAF vs NO-RACAF — Improved Training

Trains and compares RACAF and NO-RACAF under a corrected training protocol: a fixed
augmentation-RNG defect (position-independent, `(run_seed, epoch, image_id)`-keyed), a
class-weighted CORN loss, and AdamW weight decay — bundled together, applied **identically** to
both arms, across three seeds each (42, 123, 2026): six matched, independently resumable runs.

**This is NOT a modification of, or replacement for,** the finalized RACAF experiment
(`2026-09-12_02-45-05`), the finalized NO-RACAF ablation (`2026-09-13_04-11-07`), their reports,
or `colab/notebooks/stage08_corn_classifier.ipynb` / `stage08_corn_classifier_racaf_ablation.ipynb`
— none of those are read from or written to by this notebook. It writes only under
`experiments/ImprovedTraining/<EXPERIMENT_ID>/`, a separate tree.

**Audit and design:** `docs/experiments/` multi-seed improved-training audit (this session).
**New, additive modules used (no existing `.py` file is modified):** `weighted_corn.py`,
`improved_training_data.py`, `no_racaf_model.py`, `multiseed_runs.py` — see their docstrings and
`tests/test_weighted_corn.py`, `tests/test_improved_training_data.py`,
`tests/test_no_racaf_model.py`, `tests/test_multiseed_runs.py`.

**Resume is the central design requirement.** Every run can be resumed after a Colab
disconnect, a brand-new runtime, or simply re-running the training cell in the same runtime —
see `multiseed_runs.py`'s module docstring for exactly how.

**IDRiD external evaluation is explicitly out of scope for this notebook** — a separate,
later step using the existing, unmodified `colab/notebooks/idrid_external_evaluation.ipynb`,
applied to all six BEST checkpoints once this experiment is complete.


In [ ]:
# ==== [S] SESSION CONFIGURATION -- the ONLY cell to edit before a session ====
# EXPERIMENT_ID is FIXED and must never be regenerated from the current timestamp -- the whole
# resume design depends on every session naming the SAME experiment. Do not change it once any
# run under it has started.
EXPERIMENT_ID = "improved_multiseed_2026_09"

# What THIS session should do:
#   "status"              -- print the six-run status table only; nothing is built or trained.
#   "continue_next"        -- pick the first NOT_STARTED/CREATED/INTERRUPTED run, in the
#                             pre-registered order (RACAF/NO_RACAF x 42/123/2026), and train it.
#   "continue_run"         -- train a SPECIFIC run: set RUN_ARM / RUN_SEED below.
#   "evaluate_completed"   -- evaluate one COMPLETED run's BEST and LAST from disk; set RUN_ARM /
#                             RUN_SEED. Does not train.
#   "compare"              -- run the final six-run comparison. Only proceeds if all six runs are
#                             COMPLETED and every evaluation/metrics_{best,last}.json validates.
RUN_ACTION = "status"

RUN_ARM = None    # "RACAF" or "NO_RACAF" -- only read by continue_run / evaluate_completed
RUN_SEED = None   # 42, 123, or 2026      -- only read by continue_run / evaluate_completed

# Bounds how many NEW epochs this session trains before stopping and releasing the lock, even if
# the run has not reached EarlyStopping or the 50-epoch cap. None = train until this run's own
# stop condition fires (early stopping or epoch 50) or Colab disconnects. A finite budget lets
# one session deliberately leave time to also touch a second run.
SESSION_EPOCH_BUDGET = None

if RUN_ACTION not in ("status", "continue_next", "continue_run", "evaluate_completed", "compare"):
    raise ValueError(f"Unknown RUN_ACTION {RUN_ACTION!r}")
if RUN_ACTION in ("continue_run", "evaluate_completed") and (RUN_ARM is None or RUN_SEED is None):
    raise ValueError(f"RUN_ACTION={RUN_ACTION!r} requires both RUN_ARM and RUN_SEED to be set.")

print(f"EXPERIMENT_ID = {EXPERIMENT_ID!r}")
print(f"RUN_ACTION    = {RUN_ACTION!r}"
      + (f"  (arm={RUN_ARM}, seed={RUN_SEED})" if RUN_ARM else ""))


In [ ]:
# ==== [0] BOOTSTRAP -- clone/pull the repository and set sys.path ====
# `colab_config` and `setup` (imported in [0b]) live INSIDE this repository, under
# `colab/common/` -- they cannot be imported until the repository has been cloned and that
# directory is on `sys.path` (see `setup.py`'s own module docstring: "this module itself lives
# inside the repository ... so it cannot be imported until *after* the repository has been
# cloned"). This cell duplicates the exact minimal bootstrap every other stage notebook in this
# project already uses for that reason -- see `stage08_corn_classifier.ipynb` /
# `stage08_corn_classifier_racaf_ablation.ipynb`'s own "[1] BOOTSTRAP" cell. It must not import
# anything from this project (nothing from this project is importable yet).
import os
import posixpath
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)


In [ ]:
# ==== [0b] SETUP -- mount Drive, install requirements, verify the environment, imports ====
# Only NOW -- after [0] has cloned the repository and put it on `sys.path` -- can anything from
# this project be imported. `colab_setup.setup()` mounts Drive (a harmless no-op with a
# "Drive already mounted" message if [0]/an earlier cell already did so), re-verifies/pulls the
# repository (idempotent), installs requirements, and configures dataset/model environment
# variables -- identical to every other stage notebook's own "[2] SETUP" cell.
import setup as colab_setup  # colab/common/setup.py -- the project's existing, unmodified bootstrap
setup_info = colab_setup.setup()

import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)

import config
import corn
import downstream_split
import joint_training_dataset as jtd
import joint_training_model as jtm
import no_racaf_model
import weighted_corn
import improved_training_data as itd
import multiseed_runs as msr
from training import checkpointing as ckpt

print("Repository ready. multiseed_runs protocol version:", msr.PROTOCOL_VERSION)


In [ ]:
# ==== [0c] FROZEN-PATH GUARD -- this notebook must never write near a frozen artifact ====
# "ImprovedTraining" is a NEW experiment category, not one of drive_paths.PIPELINE_MODULES
# ("IQA", "VesselSegmentation", "LesionSegmentation", "FinalClassification") -- so it is built
# directly from colab_config.DRIVE.experiments_root (a plain field, always available) rather than
# through colab_config.DRIVE.experiment_dir(), which only resolves the four registered modules
# and would raise ValueError for anything else. drive_paths.py is NOT modified to register a
# fifth module -- this stays entirely additive.
EXPERIMENTS_ROOT = posixpath.join(colab_config.DRIVE.experiments_root, "ImprovedTraining")
FROZEN_FINAL_CLASSIFICATION_ROOT = colab_config.DRIVE.experiment_dir("FinalClassification")

assert posixpath.normpath(EXPERIMENTS_ROOT) != posixpath.normpath(FROZEN_FINAL_CLASSIFICATION_ROOT), (
    "ImprovedTraining and FinalClassification must be different Drive directories.")

FROZEN_RACAF_EXPERIMENT = posixpath.join(FROZEN_FINAL_CLASSIFICATION_ROOT, "2026-09-12_02-45-05")
FROZEN_NO_RACAF_EXPERIMENT = posixpath.join(FROZEN_FINAL_CLASSIFICATION_ROOT, "2026-09-13_04-11-07")
FROZEN_RACAF_ARCHIVE = posixpath.join(colab_config.FINAL_CLASSIFICATION_EXPORTED_DIR,
                                      "2026-09-12_02-45-05_BEST")
FROZEN_NO_RACAF_ARCHIVE = posixpath.join(colab_config.FINAL_CLASSIFICATION_EXPORTED_DIR,
                                         "2026-09-13_04-11-07_NO_RACAF_BEST")
FROZEN_PATHS = (FROZEN_RACAF_EXPERIMENT, FROZEN_NO_RACAF_EXPERIMENT,
               FROZEN_RACAF_ARCHIVE, FROZEN_NO_RACAF_ARCHIVE)

EXPERIMENT_ROOT_DIR = msr.experiment_root(EXPERIMENTS_ROOT, EXPERIMENT_ID)
for _frozen in FROZEN_PATHS:
    assert not posixpath.commonpath([posixpath.normpath(EXPERIMENT_ROOT_DIR),
                                     posixpath.normpath(_frozen)]) == posixpath.normpath(_frozen), (
        f"{EXPERIMENT_ROOT_DIR} resolves under the FROZEN path {_frozen} -- refusing to proceed.")
print("Frozen-path guard OK. This experiment writes only under:", EXPERIMENT_ROOT_DIR)
for _frozen in FROZEN_PATHS:
    print("  never touches:", _frozen)


In [ ]:
# ==== [1] PRE-REGISTRATION -- frozen before the first training run ====
import os

PREREGISTRATION_PATH = posixpath.join(EXPERIMENT_ROOT_DIR, msr.PREREGISTRATION_FILENAME)
EXPERIMENT_MANIFEST_PATH = posixpath.join(EXPERIMENT_ROOT_DIR, msr.EXPERIMENT_MANIFEST_FILENAME)

if not os.path.exists(PREREGISTRATION_PATH):
    if RUN_ACTION in ("continue_next", "continue_run"):
        PREREGISTRATION, PREREGISTRATION_SHA256 = msr.write_preregistration(
            PREREGISTRATION_PATH, EXPERIMENT_ID)
        print(f"Wrote a NEW PREREGISTRATION.json (sha256 {PREREGISTRATION_SHA256[:16]}...) -- "
              "this is the FIRST training run of this experiment.")
    else:
        raise RuntimeError(
            f"No PREREGISTRATION.json at {PREREGISTRATION_PATH} yet, and RUN_ACTION={RUN_ACTION!r} "
            "does not create one. Set RUN_ACTION to 'continue_next' or 'continue_run' to start "
            "this experiment, or point EXPERIMENT_ID at an experiment that already has one.")
else:
    PREREGISTRATION, PREREGISTRATION_SHA256 = msr.load_and_verify_preregistration(PREREGISTRATION_PATH)
    print(f"Loaded and verified the existing PREREGISTRATION.json (sha256 "
          f"{PREREGISTRATION_SHA256[:16]}...).")

print(f"  arms={PREREGISTRATION['arms']}  run_seeds={PREREGISTRATION['run_seeds']}  "
      f"max_epochs={PREREGISTRATION['max_epochs']}  batch_size={PREREGISTRATION['batch_size']}")
print(f"  optimizer={PREREGISTRATION['optimizer']} lr={PREREGISTRATION['learning_rate']:g} "
      f"weight_decay={PREREGISTRATION['weight_decay']}")
print(f"  class weights (grades 0-4): "
      + ", ".join(f"{w:.4f}" for w in PREREGISTRATION["class_weights"]))
print(f"  delta definition: {PREREGISTRATION['delta_definition']}")


In [ ]:
# ==== [2] VERIFY SPLIT + CACHE LOCATIONS ====
TRAIN_ENTRIES, VAL_ENTRIES, SPLIT_SHA256 = msr.verify_split()
print(f"Split verified: {len(TRAIN_ENTRIES)} train / {len(VAL_ENTRIES)} val, "
      f"sha256={SPLIT_SHA256[:16]}...")
assert SPLIT_SHA256 == msr.EXPECTED_SPLIT_SHA256

# Cache-first, exactly as the finalized RACAF/NO-RACAF notebooks: the persistent Drive cache (and
# its cache_archive/ shards) is the source of every extracted Stage 02/03/04/RACAF representation;
# [6] extracts it to local SSD once per runtime, and training reads ONLY the local copy. Nothing in
# this cell touches Drive files -- "status" and "compare" never read or extract the cache.
LOCAL_CACHE_DIR = "/content/cache/local_feature_extraction"
LOCAL_RACAF_CACHE_DIR = "/content/cache/racaf"
LOCAL_CACHE_MARKER = "/content/cache/.multiseed_archive_extracted.json"   # local disk only
LOCAL_RAW_IMAGE_DIR = "/content/cache/raw_images_for_cache_generation"  # used only by one-time generation
PERSISTENT_CACHE_DIR = config.LOCAL_FEATURE_RESULTS_DIR
PERSISTENT_RACAF_CACHE_DIR = config.RACAF_RESULTS_DIR
CACHE_ARCHIVE_DIR = os.path.join(os.path.dirname(config.LOCAL_FEATURE_RESULTS_DIR), "cache_archive")
# Deliberately nonexistent -- one-time generation takes Stage 02's live fallback, never a Drive lookup.
NO_PRECOMPUTED_STAGE02_DIR = "/content/cache/_no_precomputed_stage02_output"
print("persistent cache (Drive):", PERSISTENT_CACHE_DIR, "|", PERSISTENT_RACAF_CACHE_DIR)
print("cache archive (Drive)   :", CACHE_ARCHIVE_DIR)


In [ ]:
# ==== [3] DISCOVER EXISTING RUNS -- status table ====
STATUS_TABLE = msr.experiment_status_table(EXPERIMENTS_ROOT, EXPERIMENT_ID)
print(f"{'run':<20}{'status':<14}")
for _key, _status in STATUS_TABLE.items():
    print(f"{_key:<20}{_status:<14}")

if RUN_ACTION == "status":
    print("\nRUN_ACTION == 'status' -- nothing else runs in this session.")


In [ ]:
# ==== [4] SELECT TARGET RUN ====
TARGET_ARM = TARGET_SEED = None

if RUN_ACTION == "continue_next":
    for _arm, _seed in msr.all_run_ids():
        _key = f"{_arm}/seed_{_seed}"
        if STATUS_TABLE[_key] in (msr.STATUS_NOT_STARTED, msr.STATUS_CREATED, msr.STATUS_INTERRUPTED):
            TARGET_ARM, TARGET_SEED = _arm, _seed
            break
    if TARGET_ARM is None:
        print("No run is NOT_STARTED/CREATED/INTERRUPTED -- every run is RUNNING or COMPLETED. "
              "Nothing to continue. If a run shows RUNNING but you know that runtime is gone, use "
              "RUN_ACTION='continue_run' with force_lock=True in [7] after confirming this "
              "yourself.")
    else:
        print(f"continue_next selected: {TARGET_ARM} seed {TARGET_SEED} "
              f"(status was {STATUS_TABLE[f'{TARGET_ARM}/seed_{TARGET_SEED}']!r})")
elif RUN_ACTION in ("continue_run", "evaluate_completed"):
    TARGET_ARM, TARGET_SEED = RUN_ARM, RUN_SEED
    _key = f"{TARGET_ARM}/seed_{TARGET_SEED}"
    print(f"{RUN_ACTION} selected: {TARGET_ARM} seed {TARGET_SEED} (status {STATUS_TABLE[_key]!r})")
    if RUN_ACTION == "evaluate_completed" and STATUS_TABLE[_key] != msr.STATUS_COMPLETED:
        raise RuntimeError(f"{_key} is {STATUS_TABLE[_key]!r}, not COMPLETED -- nothing to evaluate yet.")

if TARGET_ARM is not None:
    TARGET_RUN_DIR, TARGET_CONFIG_HASH = msr.initialize_run(
        EXPERIMENTS_ROOT, EXPERIMENT_ID, TARGET_ARM, TARGET_SEED, SPLIT_SHA256,
        repo_dir=colab_config.REPO_DIR)
    print(f"Run directory: {TARGET_RUN_DIR}")
    print(f"Config hash:   {TARGET_CONFIG_HASH}")


In [ ]:
# ==== [5] BUILD / RESTORE MODEL ====
TARGET_MODEL = None
if TARGET_ARM is not None and RUN_ACTION in ("continue_next", "continue_run", "evaluate_completed"):
    TARGET_MODEL = msr.build_arm_model(TARGET_ARM, TARGET_SEED,
                                       class_weights=PREREGISTRATION["class_weights"],
                                       learning_rate=PREREGISTRATION["learning_rate"],
                                       weight_decay=PREREGISTRATION["weight_decay"])
    print(f"{TARGET_ARM} seed {TARGET_SEED}: model built and compiled (weighted CORN loss, "
          "AdamW, QWK + unweighted-CORN-loss metrics).")


In [ ]:
# ==== [6] LOCAL CACHE -- Drive cache archive -> local SSD once per runtime; training reads only this ====
# Training needs no raw image: augmentation (A) operates on the cached 8-channel Stage 05 tensor
# (canonical RGB + vessel + lesion), and RACAF's reliability is a cached value -- identical for
# both arms, for every seed. So this cell does what the finalized notebooks' own [6] did: extract
# the cache_archive/ shards (a handful of sequential Drive reads, not ~14.6k per-file opens) into
# the local cache, once per runtime. The same extracted cache serves RACAF and NO-RACAF and all
# three seeds.
#
# itd.complete_local_cache() then fills any gap ONCE: an entry only in the loose Drive cache is
# copied; an entry missing everywhere is generated by the established Phase 1 generator (its raw
# image staged for exactly that entry) and persisted to Drive. The empty-field-of-view images have
# no representation by design; they are detected once and pinned in experiment_manifest.json, so
# from then on a complete cache triggers no Drive listing, no raw image and no model load at all.
TRAIN_YIELD_ENTRIES = VAL_YIELD_ENTRIES = None
if TARGET_ARM is not None and RUN_ACTION in ("continue_next", "continue_run", "evaluate_completed"):
    import datetime
    import json
    import shutil
    import joint_cache_archive as jca

    if os.path.exists(LOCAL_CACHE_MARKER):
        print("Cache archive already extracted in this runtime -- not read again.")
    else:
        _plan = jca.plan_extraction(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                    racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
        jca.print_extraction_plan(_plan)
        if _plan["drive_unreachable"] or not _plan["fits"]:
            raise RuntimeError("Refusing to extract the cache archive: %s. Nothing was trained."
                               % ("Drive is unreachable" if _plan["drive_unreachable"]
                                  else "short by %.2f GiB" % (_plan["shortfall_bytes"] / 1024 ** 3)))
        _extract = jca.extract_archive(archive_dir=CACHE_ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                       racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                                       min_free_bytes=_plan["required_bytes"])
        jca.print_extract(_extract)
        if _extract["corrupt"] or _extract["drive_unreachable"]:
            raise RuntimeError(f"Archive extraction did not complete (corrupt: {_extract['corrupt']}, "
                               f"Drive unreachable: {_extract['drive_unreachable']}). Re-run this "
                               "cell; files already extracted are kept.")
        with open(LOCAL_CACHE_MARKER, "w") as _fh:
            json.dump({"extracted": datetime.datetime.now().isoformat(timespec="seconds")}, _fh)

    _manifest = None
    if os.path.exists(EXPERIMENT_MANIFEST_PATH):
        with open(EXPERIMENT_MANIFEST_PATH) as _fh:
            _manifest = json.load(_fh)

    CACHE_REPORT = itd.complete_local_cache(
        TRAIN_ENTRIES + VAL_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR,
        PERSISTENT_CACHE_DIR, PERSISTENT_RACAF_CACHE_DIR,
        source_image_dir=os.path.join(colab_config.APTOS2019_RAW_DIR, "train_images"),
        local_image_dir=LOCAL_RAW_IMAGE_DIR,
        known_empty_fov_ids=(_manifest or {}).get("empty_fov_ids"),
        processed_dir=NO_PRECOMPUTED_STAGE02_DIR)
    print({k: v for k, v in CACHE_REPORT.items() if k != "empty_fov_ids"})
    print(f"empty-field-of-view ids ({len(CACHE_REPORT['empty_fov_ids'])}):", CACHE_REPORT["empty_fov_ids"])

    TRAIN_YIELD_ENTRIES = itd.locally_cached_entries(TRAIN_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    VAL_YIELD_ENTRIES = itd.locally_cached_entries(VAL_ENTRIES, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    print(f"Per-epoch yield from the local cache: {len(TRAIN_YIELD_ENTRIES)} train / "
          f"{len(VAL_YIELD_ENTRIES)} val.")

    # The yield and the excluded ids are pinned once per experiment, so every run and every seed
    # trains on the identical population.
    _pin = {"n_train_yielded": len(TRAIN_YIELD_ENTRIES), "n_val_yielded": len(VAL_YIELD_ENTRIES),
            "empty_fov_ids": CACHE_REPORT["empty_fov_ids"]}
    if _manifest is not None:
        _changed = {k: (_manifest[k], v) for k, v in _pin.items() if k in _manifest and _manifest[k] != v}
        if _changed:
            raise RuntimeError(f"Training population changed since this experiment started: "
                               f"{_changed}. Investigate the cache before training any run.")
        if any(k not in _manifest for k in _pin):
            _manifest.update(_pin)
            with open(EXPERIMENT_MANIFEST_PATH, "w") as _fh:
                json.dump(_manifest, _fh, indent=2)
        print("Matches the population pinned in experiment_manifest.json.")
    elif RUN_ACTION in ("continue_next", "continue_run"):
        with open(EXPERIMENT_MANIFEST_PATH, "w") as _fh:
            json.dump({"experiment_id": EXPERIMENT_ID, "split_sha256": SPLIT_SHA256,
                       "preregistration_sha256": PREREGISTRATION_SHA256, **_pin}, _fh, indent=2)
        print(f"Pinned the training population into {EXPERIMENT_MANIFEST_PATH}.")
    else:
        raise RuntimeError("No experiment_manifest.json yet -- nothing has been trained to evaluate.")

    MIN_FREE_FOR_TRAINING_BYTES = 3 * 1024 ** 3
    _free = shutil.disk_usage("/content").free
    if _free < MIN_FREE_FOR_TRAINING_BYTES:
        raise RuntimeError(f"Refusing to start: {_free / 1024**3:.2f} GiB free on /content, "
                           f"{MIN_FREE_FOR_TRAINING_BYTES / 1024**3:.2f} GiB required.")
else:
    print(f"RUN_ACTION={RUN_ACTION!r} -- [6] does not read or extract the cache.")


In [ ]:
# ==== [7] TRAIN ====
TRAIN_OUTCOME = None
if TARGET_ARM is not None and RUN_ACTION in ("continue_next", "continue_run"):
    # Reads the local cache [6] prepared -- no Stage 03/04 model is loaded, no raw image is read.
    TRAIN_OUTCOME = msr.train_run(
        TARGET_MODEL, TARGET_RUN_DIR, TARGET_ARM, TARGET_SEED, TRAIN_YIELD_ENTRIES, VAL_YIELD_ENTRIES,
        cache_dir=LOCAL_CACHE_DIR, racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        config_hash_value=TARGET_CONFIG_HASH,
        batch_size=PREREGISTRATION["batch_size"], max_epochs=PREREGISTRATION["max_epochs"],
        early_stopping_patience=PREREGISTRATION["early_stopping"]["patience"],
        reduce_lr_patience=PREREGISTRATION["reduce_lr_on_plateau"]["patience"],
        reduce_lr_factor=PREREGISTRATION["reduce_lr_on_plateau"]["factor"],
        min_lr=PREREGISTRATION["reduce_lr_on_plateau"]["min_lr"],
        repo_dir=colab_config.REPO_DIR, session_epoch_budget=SESSION_EPOCH_BUDGET, verbose=1)

    print(f"\n{TARGET_ARM} seed {TARGET_SEED}: completed_epoch={TRAIN_OUTCOME.completed_epoch}  "
          f"stopped={TRAIN_OUTCOME.stopped} ({TRAIN_OUTCOME.stop_reason})  "
          f"epochs_trained_this_call={TRAIN_OUTCOME.epochs_trained_this_call}")
    if not TRAIN_OUTCOME.stopped:
        print("This run is not finished -- re-run [S]-[7] (a fresh runtime is fine) to continue "
              f"from epoch {TRAIN_OUTCOME.completed_epoch + 1}.")
else:
    print(f"RUN_ACTION={RUN_ACTION!r} -- [7] does not train.")


In [ ]:
# ==== [8] EVALUATE CURRENT RUN -- BEST and LAST, loaded fresh from Drive ====
EVALUATION_RESULT = None
if TARGET_ARM is not None and RUN_ACTION in ("evaluate_completed",) or (
        TARGET_ARM is not None and RUN_ACTION in ("continue_next", "continue_run")
        and TRAIN_OUTCOME is not None and TRAIN_OUTCOME.stopped):
    _eval_model = msr.build_arm_model(TARGET_ARM, TARGET_SEED,
                                      class_weights=PREREGISTRATION["class_weights"],
                                      learning_rate=PREREGISTRATION["learning_rate"],
                                      weight_decay=PREREGISTRATION["weight_decay"], verbose=0)

    EVALUATION_RESULT = {}
    for _which in ("best", "last"):
        if _which == "best":
            _slot_dir, _pointer = msr.read_best(TARGET_RUN_DIR)
            if _slot_dir is None:
                print("No BEST published yet for this run -- skipping BEST evaluation.")
                continue
            _epoch = _pointer["epoch"]
            ckpt.load_model_weights_only(_eval_model, os.path.join(_slot_dir, ckpt.MODEL_WEIGHTS_FILENAME))
        else:
            _generation_dir = ckpt.find_resumable_generation(os.path.join(TARGET_RUN_DIR, "checkpoints"))
            if _generation_dir is None:
                print("No valid LAST generation found -- skipping LAST evaluation.")
                continue
            _last_state = ckpt.read_state(_generation_dir)
            _epoch = _last_state.completed_epoch
            ckpt.load_model_weights_only(_eval_model, os.path.join(_generation_dir, ckpt.MODEL_WEIGHTS_FILENAME))

        _rows = msr.evaluate_arm_from_disk(
            _eval_model, VAL_YIELD_ENTRIES, cache_dir=LOCAL_CACHE_DIR,
            racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
        _metrics = msr.summarize_predictions(_rows)
        _metrics["epoch"] = _epoch
        EVALUATION_RESULT[_which] = {"rows": _rows, "metrics": _metrics}

        import csv
        _eval_dir = os.path.join(TARGET_RUN_DIR, "evaluation")
        os.makedirs(_eval_dir, exist_ok=True)
        with open(os.path.join(_eval_dir, f"per_sample_{_which}.csv"), "w", newline="") as _fh:
            if _rows:
                _writer = csv.DictWriter(_fh, fieldnames=list(_rows[0].keys()))
                _writer.writeheader()
                _writer.writerows(_rows)
        with open(os.path.join(_eval_dir, f"metrics_{_which}.json"), "w") as _fh:
            import json
            json.dump(_metrics, _fh, indent=2)
        print(f"{TARGET_ARM} seed {TARGET_SEED} {_which.upper()} (epoch {_epoch}): "
              f"QWK={_metrics.get('qwk'):.4f}  val_loss/unweighted n/a here (see history/)  "
              f"acc={_metrics.get('accuracy'):.4f}  n={_metrics.get('n')}")
else:
    print("Nothing to evaluate in this session (run not finished, or RUN_ACTION does not call for it).")


In [ ]:
# ==== [9] CONTINUE NEXT / STATUS (refresh after this session's work) ====
STATUS_TABLE = msr.experiment_status_table(EXPERIMENTS_ROOT, EXPERIMENT_ID)
print(f"{'run':<20}{'status':<14}")
for _key, _status in STATUS_TABLE.items():
    print(f"{_key:<20}{_status:<14}")
_remaining = [k for k, v in STATUS_TABLE.items() if v != msr.STATUS_COMPLETED]
if _remaining:
    print(f"\n{len(_remaining)} run(s) not yet COMPLETED: {_remaining}. Re-run this notebook "
          "(any runtime) with RUN_ACTION='continue_next' to keep going.")
else:
    print("\nAll six runs are COMPLETED. Set RUN_ACTION='compare' and re-run [S] onward for the "
          "final six-run comparison.")


In [ ]:
# ==== [X] FINAL SIX-RUN COMPARISON -- only once every run is COMPLETED and valid ====
if RUN_ACTION == "compare":
    import numpy as np

    STATUS_TABLE = msr.experiment_status_table(EXPERIMENTS_ROOT, EXPERIMENT_ID)
    _not_done = [k for k, v in STATUS_TABLE.items() if v != msr.STATUS_COMPLETED]
    if _not_done:
        raise RuntimeError(f"Not every run is COMPLETED yet: {_not_done}. The six-run comparison "
                           "refuses to run on a partial experiment.")

    PER_RUN = {}
    for _arm, _seed in msr.all_run_ids():
        _run_dir = msr.run_dir(EXPERIMENTS_ROOT, EXPERIMENT_ID, _arm, _seed)
        _best_metrics_path = os.path.join(_run_dir, "evaluation", "metrics_best.json")
        _last_metrics_path = os.path.join(_run_dir, "evaluation", "metrics_last.json")
        if not (os.path.exists(_best_metrics_path) and os.path.exists(_last_metrics_path)):
            raise RuntimeError(f"{_run_dir}: run [8] (evaluate_completed) for this run before "
                               "comparing -- its evaluation/metrics_{best,last}.json are missing.")
        import json
        with open(_best_metrics_path) as _fh: _best = json.load(_fh)
        with open(_last_metrics_path) as _fh: _last = json.load(_fh)
        with open(os.path.join(_run_dir, "evaluation", "per_sample_best.csv")) as _fh:
            import csv
            _best_rows = list(csv.DictReader(_fh))
        _run_history = msr.read_history(_run_dir)
        PER_RUN[(_arm, _seed)] = {"best": _best, "last": _last, "best_rows": _best_rows,
                                 "history": _run_history}
        _best_epoch, _best_qwk = _best["epoch"], _best["qwk"]
        _last_epoch, _last_qwk = _last["epoch"], _last["qwk"]
        print(f"{_arm} seed {_seed}: BEST epoch {_best_epoch} QWK {_best_qwk:.4f} | "
              f"LAST epoch {_last_epoch} QWK {_last_qwk:.4f} | "
              f"{len(_run_history)} epochs trained")

    print("\n=== ACROSS-SEED SUMMARY (BEST val_QWK) ===")
    for _arm in msr.ARMS:
        _values = [PER_RUN[(_arm, s)]["best"]["qwk"] for s in msr.RUN_SEEDS]
        print(f"{_arm}: mean={np.mean(_values):.4f} sd={np.std(_values):.4f} "
              f"median={np.median(_values):.4f} min={np.min(_values):.4f} max={np.max(_values):.4f}  "
              f"per-seed={dict(zip(msr.RUN_SEEDS, _values))}")

    print("\n=== PAIRED RACAF - NO_RACAF (Delta = QWK_RACAF - QWK_NO_RACAF) ===")
    DELTAS = {}
    for _seed in msr.RUN_SEEDS:
        _racaf_rows = {r["image_id"]: r for r in PER_RUN[("RACAF", _seed)]["best_rows"]}
        _no_racaf_rows = {r["image_id"]: r for r in PER_RUN[("NO_RACAF", _seed)]["best_rows"]}
        _shared_ids = sorted(set(_racaf_rows) & set(_no_racaf_rows))
        if len(_shared_ids) != len(_racaf_rows) or len(_shared_ids) != len(_no_racaf_rows):
            raise RuntimeError(f"seed {_seed}: RACAF and NO_RACAF evaluated different image sets "
                               f"({len(_racaf_rows)} vs {len(_no_racaf_rows)}, {len(_shared_ids)} "
                               "shared) -- refusing to compute a paired comparison on mismatched images.")
        _y_true = np.array([int(_racaf_rows[i]["true_grade"]) for i in _shared_ids])
        _y_true_check = np.array([int(_no_racaf_rows[i]["true_grade"]) for i in _shared_ids])
        if not np.array_equal(_y_true, _y_true_check):
            raise RuntimeError(f"seed {_seed}: ground truth disagrees between arms for some image.")
        _y_racaf = np.array([int(_racaf_rows[i]["predicted_grade"]) for i in _shared_ids])
        _y_no_racaf = np.array([int(_no_racaf_rows[i]["predicted_grade"]) for i in _shared_ids])

        from sklearn.metrics import cohen_kappa_score
        _qwk_racaf = cohen_kappa_score(_y_true, _y_racaf, weights="quadratic")
        _qwk_no_racaf = cohen_kappa_score(_y_true, _y_no_racaf, weights="quadratic")
        _delta = _qwk_racaf - _qwk_no_racaf
        DELTAS[_seed] = _delta

        _rng = np.random.default_rng(20260913)
        _boot = []
        for _ in range(2000):
            _idx = _rng.integers(0, len(_shared_ids), size=len(_shared_ids))
            _qa = cohen_kappa_score(_y_true[_idx], _y_racaf[_idx], weights="quadratic")
            _qb = cohen_kappa_score(_y_true[_idx], _y_no_racaf[_idx], weights="quadratic")
            if np.isfinite(_qa) and np.isfinite(_qb):
                _boot.append(_qa - _qb)
        _lo, _hi = np.percentile(_boot, [2.5, 97.5]) if _boot else (float("nan"), float("nan"))
        print(f"seed {_seed}: QWK_RACAF={_qwk_racaf:.4f} QWK_NO_RACAF={_qwk_no_racaf:.4f} "
              f"Delta={_delta:+.4f}  bootstrap 95% CI [{_lo:+.4f}, {_hi:+.4f}]  n={len(_shared_ids)}")

    _mean_delta = float(np.mean(list(DELTAS.values())))
    _all_positive = all(d > 0 for d in DELTAS.values())
    _all_negative = all(d < 0 for d in DELTAS.values())
    print(f"\nmean Delta = {_mean_delta:+.4f}   per-seed sign consistency: "
          f"{sum(1 for d in DELTAS.values() if d > 0)}/3 positive, "
          f"{sum(1 for d in DELTAS.values() if d < 0)}/3 negative")

    if _mean_delta > 0 and _all_positive:
        VERDICT = "SUPPORT_RACAF (pending bootstrap-CI check in >=2/3 seeds -- see printed CIs above)"
    elif _mean_delta < 0 and _all_negative:
        VERDICT = "SUPPORT_NO_RACAF (pending bootstrap-CI check in >=2/3 seeds -- see printed CIs above)"
    else:
        VERDICT = "INCONCLUSIVE"
    print(f"\nPre-registered mechanical verdict (sign-consistency stage): {VERDICT}")
    print("This is the primary comparison. Secondary metrics (severe-grade recall, duplicate-"
          "excluded, calibration, McNemar) do not override it -- see the [R] report cell.")
else:
    print(f"RUN_ACTION={RUN_ACTION!r} -- [X] only runs when RUN_ACTION == 'compare'.")


In [ ]:
# ==== [R] FINAL REPORT (skeleton -- fills in once [X] has run) ====
if RUN_ACTION == "compare":
    print("Six-run comparison complete. Transfer the printed [X] output into "
          "docs/experiments/Multiseed_Improved_Training_Report.md's Results sections "
          "(6-14) by hand, exactly as earlier experiment reports in this project were filled in "
          "from their own notebooks' saved outputs -- this cell does not write repository files.")
else:
    print(f"RUN_ACTION={RUN_ACTION!r} -- [R] has nothing to report yet.")
